# How to traverse to GO children and descendants

Traverse immediate children or all descendants with or without user-specified optional relationships


  * **Children and Descendants described**
  * **Code to get Children and Descendants**
    * **Get children through *is_a* relationship**
    * **Get children through *is_a* relationship and optional relationship, *part_of*.**
    * **Get descendants through *is_a* relationship**
    * **Get descendants through *is_a* relationship and optional relationship, *part_of*.**

## Children and Descendants

### Children

Children are terms directly below a GO term

### Descendants

Descendants are all terms below a GO term in the hierarchy

In [ ]:
import os
from goatools.obo_parser import GODag

# Load a small test GO DAG and all the optional relationships,
# like 'regulates' and 'part_of'
godag = GODag('../tests/data/heartjogging.obo',
              optional_attrs={'relationship'})

#### Get children through *is_a* relationship

Direct children through is_a relationship only

In [ ]:
GO_ID = 'GO:0003143'  # embryonic heart tube morphogenesis
goterm = godag[GO_ID]

# Get direct children through is_a
children_isa = [child.item_id for child in goterm.children]
print('{GO} children (is_a): {C}'.format(
    GO=GO_ID,
    C=children_isa))

#### Get children through *is_a* relationship and optional relationship, *part_of*

Direct children through is_a and part_of relationships

In [ ]:
from goatools.godag.go_tasks import get_go2children

optional_relationships = {"part_of"}
# Note: GODag inherits from dict, so we pass godag directly
go2children_partof = get_go2children(godag, optional_relationships)
print("{GO} children (is_a + part_of): {C}".format(
    GO=GO_ID,
    C=go2children_partof.get(GO_ID, set())))

#### Get descendants through *is_a* relationship

All descendants through is_a only

In [ ]:
# Using GOTerm method
descendants_isa = goterm.get_all_children()
print('{GO} descendants (is_a): {D}'.format(
    GO=GO_ID,
    D=sorted(descendants_isa)))

#### Get descendants through *is_a* relationship and optional relationship, *part_of*

All descendants through is_a and part_of relationships

In [ ]:
from goatools.godag.go_tasks import get_go2descendants

go2descendants_partof = get_go2descendants(godag.values(), optional_relationships)
print('{GO} descendants (is_a + part_of): {D}'.format(
    GO=GO_ID,
    D=sorted(go2descendants_partof.get(GO_ID, set()))))

#### Alternative: Use GOTerm.get_all_lower() for all relationships

This method automatically includes all relationship types

In [ ]:
# Get all descendants through is_a and ALL relationships
all_lower = goterm.get_all_lower()
print('{GO} descendants (is_a + all relationships): {D}'.format(
    GO=GO_ID,
    D=sorted(all_lower)))

# Show the additional descendants found through relationships
additional = all_lower - descendants_isa
if additional:
    print("\nAdditional descendants through relationships:")
    for goid in sorted(additional):
        desc_term = godag[goid]
        print(f"  {goid}: {desc_term.name}")

## Summary

To get descendants with relationships in GOAtools:

1. **Load GO DAG with relationships**: Use `optional_attrs={'relationship'}` when loading
2. **For a single GO term**:
   - `goterm.get_all_children()` - descendants through is_a only
   - `goterm.get_all_lower()` - descendants through is_a + all relationships
3. **For multiple GO terms (batch processing)**:
   - `get_go2descendants(godag.values(), relationships=None)` - is_a only
   - `get_go2descendants(godag.values(), relationships=True)` - is_a + all relationships
   - `get_go2descendants(godag.values(), relationships={'part_of'})` - is_a + selected relationships
4. **For hierarchy visualization**: Use `wr_hier.py -r` command-line tool